# Home task: pandas 

## Question 1

- Load the energy data from the file [Energy Indicators.xls](http://unstats.un.org/unsd/environment/excel_file_tables/2013/Energy%20Indicators.xls).
It is a list of indicators of energy supply and renewable electricity production from the United Nations for the year 2013.


- It should be put into a DataFrame with the variable name of "energy"


- Make sure to exclude the footer and header information from the datafile.


- The first two columns are unneccessary, so you should get rid of them, and you should change the column labels so that the columns are:<br>
`['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']`


- Convert `Energy Supply` to gigajoules (there are 1,000,000 gigajoules in a petajoule).


- For all countries which have missing data (e.g. data with `...`) make sure this is reflected as `np.NaN` values.


- Rename the following list of countries (for use in later questions):
    - `Republic of Korea`: `South Korea`,
    - `United States of America`: `United States`,
    - `United Kingdom of Great Britain and Northern Ireland`: `United Kingdom`,
    - `China, Hong Kong Special Administrative Region`: `Hong Kong`


- There are also several countries with numbers and/or parenthesis in their name. Be sure to remove these, e.g.:
    - `Bolivia (Plurinational State of)` should be `Bolivia`,
    - `Switzerland17` should be `Switzerland`.


- Next, load the GDP data from the file ["world_bank.csv"](http://data.worldbank.org/indicator/NY.GDP.MKTP.CD). 
It is a csv containing countries' GDP from 1960 to 2015 from World Bank. Call this DataFrame "GDP"


- Make sure to skip the header, and rename the following list of countries:
    - `Korea, Rep.`: `South Korea`,
    - `Iran, Islamic Rep.`: `Iran`,
    - `Hong Kong SAR, China`: `Hong Kong`


- Finally, load the "Sciamgo Journal and Country Rank data for [Energy Engineering and Power Technology"](http://www.scimagojr.com/countryrank.php?category=2102). It ranks countries based on their journal contributions in the aforementioned area. Call this DataFrame "ScimEn"


- Join the three datasets: Energy, GDP, and ScimEn into a new dataset (using the intersection of country names). Use only the 10 years (2006-2015) of GDP data and only the top 15 countries by Scimagojr 'Rank' (Rank 1 through 15).


- The index of this DataFrame should be the name of the country, and the columns should be<br>
`['Rank', 'Documents', 'Citable documents', 'Citations', 'Self-citations', 'Citations per document', 'H index', 'Energy Supply', 'Energy Supply per Capita', '% Renewable', '2006', '2007', '2008', '2009', '2010', '2011', 2012', '2013', '2014', '2015']`

Function "answer_one" should return the resulted DataFrame (20 columns and 15 entries)

In [173]:
import pandas as pd
import numpy as np

In [174]:
def load_energy_data():
    energy = pd.read_excel("Energy Indicators.xls", skiprows=17, skipfooter=38)
    
    energy.drop(columns=["Unnamed: 0", "Unnamed: 1"], inplace=True)

    energy.rename(columns={
        "Unnamed: 2": "Country",
        "Petajoules": "Energy Supply",
        "Gigajoules": "Energy Supply per Capita",
        "%": "% Renewable"
    }, inplace=True)

    energy.replace("...", np.nan, inplace=True)

    energy["Energy Supply"] *= 1_000_000

    energy["Country"] = (
        energy["Country"]
        .str.replace(r"\(.*?\)", "", regex=True)    #Delete all parenthesis and information in them
        .str.replace(r"\d+", "", regex=True)        #Delete all numbers
        ).str.strip()
    
    energy["Country"] = energy["Country"].replace({"Republic of Korea" : "South Korea",
        "United States of America" : "United States",
        "United Kingdom of Great Britain and Northern Ireland" : "United Kingdom",
        "China, Hong Kong Special Administrative Region" : "Hong Kong"})
    
    return energy


In question 1 it says that world_bank is .csv, however it's not

In [175]:
def load_GDP():
    GDP = pd.read_excel("world_bank.xls", skiprows=3)

    GDP["Country Name"] = GDP["Country Name"].replace({
        "Korea, Rep.": "South Korea",
        "Iran, Islamic Rep.": "Iran",
        "Hong Kong SAR, China": "Hong Kong"
    })

    GDP.rename(columns={
        "Country Name": "Country"
    }, inplace= True)
    
    return GDP


In [176]:
def answer_one():
    energy = load_energy_data().set_index("Country")

    GDP = (load_GDP()
           .set_index("Country")
           .loc[:, "2006":"2015"])

    scimEn = (pd.read_excel("scimagojr.xlsx")
              .drop(columns="Region"))

    coolDF = (scimEn[:15]
              .set_index("Country")
              .join(energy, how="inner")
              .join(GDP, how="inner"))

    return coolDF

In [177]:
answer = answer_one()
print(answer.info())
answer

<class 'pandas.DataFrame'>
Index: 15 entries, China to Australia
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Rank                      15 non-null     int64  
 1   Documents                 15 non-null     int64  
 2   Citable documents         15 non-null     int64  
 3   Citations                 15 non-null     int64  
 4   Self-citations            15 non-null     int64  
 5   Citations per document    15 non-null     float64
 6   H index                   15 non-null     int64  
 7   Energy Supply             15 non-null     object 
 8   Energy Supply per Capita  15 non-null     object 
 9   % Renewable               15 non-null     float64
 10  2006                      15 non-null     float64
 11  2007                      15 non-null     float64
 12  2008                      15 non-null     float64
 13  2009                      15 non-null     float64
 14  2010             

,Rank,Documents,Citable documents,Citations,Self-citations,Citations per document,H index,Energy Supply,Energy Supply per Capita,% Renewable,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,,,,,,,,,,,,,,,,,,,,
China,1,273437,272374,2336764,1615239,8.55,245,127191000000,93,19.754910,2.752119e+12,3.550328e+12,4.594337e+12,5.101691e+12,6.087192e+12,7.551545e+12,8.532186e+12,9.570471e+12,1.047562e+13,1.106157e+13
United States,2,175891,172431,2230544,724472,12.68,363,90838000000,286,11.570980,1.381559e+13,1.447423e+13,1.476986e+13,1.447806e+13,1.504896e+13,1.559973e+13,1.625397e+13,1.684319e+13,1.755068e+13,1.820602e+13
India,3,55082,53775,463165,162944,8.41,181,33195000000,26,14.969080,9.402599e+11,1.216736e+12,1.198895e+12,1.341888e+12,1.675616e+12,1.823052e+12,1.827638e+12,1.856721e+12,2.039126e+12,2.103588e+12
Japan,4,50523,50065,488062,119930,9.66,193,18984000000,149,10.232820,4.601663e+12,4.579750e+12,5.106679e+12,5.289494e+12,5.759072e+12,6.233147e+12,6.272363e+12,5.212328e+12,4.896994e+12,4.444931e+12
United Kingdom,5,43389,42284,615670,111290,14.19,226,7920000000,124,10.600470,2.709978e+12,3.092996e+12,2.931684e+12,2.417566e+12,2.491397e+12,2.666403e+12,2.706341e+12,2.786315e+12,3.065223e+12,2.934858e+12
Germany,6,38739,38013,433148,95145,11.18,196,13261000000,165,17.901530,2.994704e+12,3.425578e+12,3.745264e+12,3.411261e+12,3.399668e+12,3.749315e+12,3.527143e+12,3.733805e+12,3.889093e+12,3.357586e+12
Russian Federation,7,36735,36560,115938,54993,3.16,90,30709000000,214,17.288680,9.899321e+11,1.299703e+12,1.660848e+12,1.222646e+12,1.524917e+12,2.045923e+12,2.208294e+12,2.292470e+12,2.059242e+12,1.363482e+12
Canada,8,33472,32863,568080,100953,16.97,227,10431000000,296,61.945430,1.319265e+12,1.468820e+12,1.552990e+12,1.374625e+12,1.617343e+12,1.793327e+12,1.828366e+12,1.846597e+12,1.805750e+12,1.556509e+12
Italy,9,27983,26940,352993,87828,12.61,166,6530000000,109,33.667230,1.949552e+12,2.213102e+12,2.408655e+12,2.199929e+12,2.136100e+12,2.294994e+12,2.086958e+12,2.141924e+12,2.162010e+12,1.836638e+12


## Answer the following questions in the context of only the top 15 countries by Scimagojr Rank (aka the DataFrame returned by `answer_one()`)

### Question 2
What is the average GDP over the last 10 years for each country? (exclude missing values from this calculation.)

*This function should return a Series named `avgGDP` with 15 countries and their average GDP sorted in descending order.*

In [178]:
def answer_two():
    Top15 = answer_one()

    avgGDP = (Top15.loc[:, "2006":"2015"]
              .mean(axis=1, skipna=True)
              .sort_values(ascending=False))
    
    return avgGDP

In [179]:
answer_two()

Country
United States         1.570403e+13
China                 6.927707e+12
Japan                 5.239642e+12
Germany               3.523342e+12
United Kingdom        2.780276e+12
France                2.691337e+12
Italy                 2.142986e+12
Brazil                1.988889e+12
Russian Federation    1.666746e+12
Canada                1.616359e+12
India                 1.602352e+12
Spain                 1.400886e+12
South Korea           1.221372e+12
Australia             1.207513e+12
Iran                  4.563261e+11
dtype: float64

### Question 3
By how much had the GDP changed over the 10 year span for the country with the 6th largest average GDP?

*This function should return a single number.*

In [180]:
def answer_three():
    Top15 = answer_one()
    country = answer_two().index[5] #Get index(Name) by it's position in the series
    change = Top15["2015"].loc[country] - Top15["2006"].loc[country]
    return change

In [181]:
answer_three()

np.float64(118652421857.7959)

### Question 4

Create a new column that is the ratio of Self-Citations to Total Citations. 
What is the maximum value for this new column, and what country has the highest ratio?

*This function should return a tuple with the name of the country and the ratio.*

In [182]:
def answer_four():
    Top15 = answer_one()
    Top15["Citation ratio"] = Top15["Self-citations"]/Top15["Citations"]
    max_ratio = Top15["Citation ratio"].max()
    country_name = Top15["Citation ratio"].idxmax()
    return (country_name, round(float(max_ratio), 5))


In [183]:
answer_four()

('China', 0.69123)

### Question 5

Create a column that estimates the population using Energy Supply and Energy Supply per capita. 
What is the third most populous country according to this estimate?

*This function should return a single string value.*

In [184]:
def answer_five():
    Top15 = answer_one()
    Top15["Estimated Population"] = Top15["Energy Supply"]/Top15["Energy Supply per Capita"]
    country = pd.to_numeric(Top15["Estimated Population"]).nlargest(3).index[-1]
    return country

In [185]:
answer_five()

'United States'

### Question 6
Create a column that estimates the number of citable documents per person. 
What is the correlation between the number of citable documents per capita and the energy supply per capita? Use the `.corr()` method, (Pearson's correlation).

*This function should return a single number.*


In [ ]:
def answer_six():
    Top15 = answer_one()
    Top15["Estimated Population"] = Top15["Energy Supply"]/Top15["Energy Supply per Capita"]
    Top15["Citable docs per person"] = Top15["Citable documents"]/Top15["Estimated Population"]
    correlation = Top15["Energy Supply per Capita"].corr(Top15["Citable docs per person"], "pearson")
    return correlation


In [187]:
answer_six()  #We see high correlation ~0.74

np.float64(0.7434709127726777)

### Question 7
Use the following dictionary to group the Countries by Continent, then create a dateframe that displays the sample size (the number of countries in each continent bin), and the sum, mean, and std deviation for the estimated population of each country.

```python
ContinentDict  = {'China':'Asia', 
                  'United States':'North America', 
                  'Japan':'Asia', 
                  'United Kingdom':'Europe', 
                  'Russian Federation':'Europe', 
                  'Canada':'North America', 
                  'Germany':'Europe', 
                  'India':'Asia',
                  'France':'Europe', 
                  'South Korea':'Asia', 
                  'Italy':'Europe', 
                  'Spain':'Europe', 
                  'Iran':'Asia',
                  'Australia':'Australia', 
                  'Brazil':'South America'}
```

*This function should return a DataFrame with index named Continent `['Asia', 'Australia', 'Europe', 'North America', 'South America']` and columns `['size', 'sum', 'mean', 'std']`*

In [188]:
def answer_seven():
    Top15 = answer_one()
    ContinentDict  = {'China':'Asia', 
                  'United States':'North America', 
                  'Japan':'Asia', 
                  'United Kingdom':'Europe', 
                  'Russian Federation':'Europe', 
                  'Canada':'North America', 
                  'Germany':'Europe', 
                  'India':'Asia',
                  'France':'Europe', 
                  'South Korea':'Asia', 
                  'Italy':'Europe', 
                  'Spain':'Europe', 
                  'Iran':'Asia',
                  'Australia':'Australia', 
                  'Brazil':'South America'}
    Top15["Estimated Population"] = (Top15["Energy Supply"]/Top15["Energy Supply per Capita"]).round()

    continent = Top15['Estimated Population'].groupby(ContinentDict).agg(["size","sum", "mean", "std"])
    continent.index.name = "Continents"
    
    
    return continent

In [189]:
answer_seven()

,size,sum,mean,std
Continents,,,,
Asia,5,2898666386.0,579733277.2,6.790979e+08
Australia,1,23316017.0,23316017.0,NaN
Europe,6,457929667.0,76321611.166667,3.464767e+07
North America,2,352855250.0,176427625.0,1.996696e+08
South America,1,205915254.0,205915254.0,NaN
